In [ ]:
class ScratchBiGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru       = nn.GRU(embed_dim, embed_dim, batch_first=True, bidirectional=True)
        self.dropout   = nn.Dropout(0.3)
        self.scorer    = nn.Sequential(
            nn.Linear(embed_dim * 4, embed_dim),   # *4: BiGRU doubles, 2 texts joined
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, 1),
        )

    def encode(self, x):
        mask   = (x != 0).float().unsqueeze(-1)    # ignore padding
        out, _ = self.gru(self.embedding(x))
        return (out * mask).sum(1) / mask.sum(1).clamp(min=1)   # masked mean pool

    def forward(self, prompt_ids, option_ids):
        B, C, L = option_ids.shape
        p_vec  = self.encode(prompt_ids)
        o_vec  = self.encode(option_ids.reshape(B*C, L)).reshape(B, C, -1)
        joined = torch.cat([p_vec.unsqueeze(1).expand(-1, C, -1), o_vec], dim=-1)
        return self.scorer(self.dropout(joined)).squeeze(-1)     # (B, 5)

scratch_model = ScratchBiGRU(len(vocab)).to(DEVICE)   # 644,737 parameters